# Uni-Projekt: Large Language Models from Scratch
**Gruppe:** Phillip, Konstantin, Fabian

**Ziel:** Aufbau und Training eines eigenen LLM-Modells, mithilfe eines Datensatzes der Kochrezepte enthält.

# Datensatz einlesen
* Den CSV Datensatz einlsesn
* Die ersten Zeilen ausgeben

In [4]:
import os
import pandas as pd

file_path = "full_dataset.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    # DataFrame auf die ersten 1000 Zeilen beschränken --> effitienteres entwickeln
    df = df.head(1000)
    print(df.info())
    print(df.head())   
else:    print(f"File '{file_path}' does not exist.")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   title        1000 non-null   object
 2   ingredients  1000 non-null   object
 3   directions   1000 non-null   object
 4   link         1000 non-null   object
 5   source       1000 non-null   object
 6   NER          1000 non-null   object
dtypes: int64(1), object(6)
memory usage: 54.8+ KB
None
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg...   
3  ["

- Die CSV-Datei als einen langen Sring formatieren

In [5]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients:{ingredients}\nDirections:{directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"Der gesamte Datensatz wurde in einen String umgewandelt.")
print(f"Gesamtanzahl der Zeichen: {len(raw_text)}")
print("\nVorschau der ersten 300 Zeichen:")
print(raw_text[:300])

    

Der gesamte Datensatz wurde in einen String umgewandelt.
Gesamtanzahl der Zeichen: 470945

Vorschau der ersten 300 Zeichen:
Recepie: No-Bake Nut Cookies
Ingredients:1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions:In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and bu


# Tokenizer bauen


- Tokenizer auf mit einfachen Textbeispielen bauen und später auf den Datensatz anwenden
- Mithilfe von RegEx die Wörter trennen

In [6]:
import re

text = "This is a sample text: with numbers 123, special characters !@# and a few more.  ."
def split_text(text):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [token for token in tokens if token.strip()]
    return tokens


result = split_text(text)


print(result)

['This', 'is', 'a', 'sample', 'text', ':', 'with', 'numbers', '123', ',', 'special', 'characters', '!', '@#', 'and', 'a', 'few', 'more', '.', '.']


- Trennung der Wörter auf den Datensatz anwenden

In [7]:
split = split_text(raw_text)
print(f"Anzahl der Wörter: {len(split)}")
print(split[:20])

Anzahl der Wörter: 110664
['Recepie', ':', 'No-Bake', 'Nut', 'Cookies', 'Ingredients', ':', '1', 'c', '.', 'firmly', 'packed', 'brown', 'sugar', ',', '1/2', 'c', '.', 'evaporated', 'milk']


# Wort-Tokens in Tokens Ids konvertieren

- Vokabular das alle Wort-Tokens im Datensatzenthält

In [8]:
all_words = sorted(set(split))
vocab_size = len(all_words)

print(vocab_size)

3738


- Jedem Token eine ID zuweisen

In [9]:
vocab = {token:integer for integer,token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
('#1', 2)
('#2', 3)
('%', 4)
('&', 5)
("'", 6)
('(', 7)
(')', 8)
('*', 9)
('*Apricot', 10)
('*Can', 11)
('*I', 12)
('+', 13)
('+/-', 14)
('+cooking', 15)
(',', 16)
('-', 17)
('--', 18)
('.', 19)
('0', 20)
('0%', 21)
('1', 22)
('1%', 23)
('1-', 24)
('1-1/2', 25)
('1-8', 26)
('1-gallon', 27)
('1-inch', 28)
('1-quart', 29)
('1/16', 30)
('1/2', 31)
('1/2-inch', 32)
('1/2-pint', 33)
('1/2-quart', 34)
('1/2\\', 35)
('1/2\\tcup', 36)
('1/3', 37)
('1/4', 38)
('1/4-inch', 39)
('1/4\\tturn', 40)
('1/8', 41)
('1/8-inch', 42)
('10', 43)
('10-inch', 44)
('100', 45)
('100%', 46)
('10x', 47)
('11', 48)
('112', 49)
('12', 50)


- Tokenizer Klasse bauen mit der sowohl encoded (Text zu Ids) als auch decoded (Ids zu Texten) werden kann

In [10]:
class TokenizerV1:
    def __init__(self, vocab):
        self.int_to_str = {integer: token for token, integer in vocab.items()}
        self.str_to_int = vocab
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
    

- Jetzt können alle im Vokabular enthaltenen tokens in beliebiger Reihenfolge übergeben werden und in Ids umgewandelt werden
- Ebenfalls kann eine Folge von Ids zurück in Tokens und damit in Text umgewandelt werden

In [11]:
tokenizer = TokenizerV1(vocab)
encoded = tokenizer.encode("baked brown sugar Cookies")
decoded = tokenizer.decode([500, 401, 602, 1503, 95])
print(encoded)
print(decoded)

[1694, 1820, 3436, 457]
Crush Children English Very 270\u00b0


- UNK Token für unbekannte Wörter und end_of_text Token einfügen

In [12]:
all_tokens = sorted(list(set(all_words)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

len(vocab.items())

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('youre', 3735)
('zeppole', 3736)
('zucchini', 3737)
('<|endoftext|>', 3738)
('<|unk|>', 3739)


- Tokenizer anpassen, damit er die neuen Tokens "<|endoftext|>" und "<|unk|>" korrekt verarbeitet

In [13]:
class TokenizerV2:
    def __init__(self, vocab):
        self.int_to_str = {integer: token for token, integer in vocab.items()}
        self.str_to_int = vocab
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]                        
        preprocessed = [
            item if item.strip() in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- Neuen Tokenizer testen

In [14]:
tokenizer_2 = TokenizerV2(vocab)

text1 = "This is a sample text: with numbers 123, special characters !@# and a few more."
text2 = " And a few more Recepie words and Ingredients Apple, sugar, honey car."

text = " <|endoftext|> ".join([text1, text2])

encoded_v2 = tokenizer_2.encode(text)
print(encoded_v2)

decoded_v2 = tokenizer_2.decode(encoded_v2)
print(decoded_v2)

[1442, 2517, 1598, 3739, 3739, 182, 3708, 3739, 3739, 16, 3351, 3739, 0, 3739, 1650, 1598, 2268, 2744, 19, 3738, 208, 1598, 2268, 2744, 1176, 3739, 1650, 783, 214, 16, 3436, 16, 2480, 3739, 19]
This is a <|unk|> <|unk|> : with <|unk|> <|unk|>, special <|unk|>! <|unk|> and a few more. <|endoftext|> And a few more Recepie <|unk|> and Ingredients Apple, sugar, honey <|unk|>.


## BytePair encoding

- Der einfache Tokeniszer wird durch BytePair encoding ersetzt
- Das erlaubt dem Model Wörter zu unterteilen (sogar bis auf den einzelenen Buchstaben) um neue Wörte die nicht im initial geladenen Vakabular waren in Token Ids zu verwandeln

In [15]:
import tiktoken
import importlib

print ("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


- Tokenizer mit tiktoken erstellen und decoden/encoden

In [16]:
tiktokenizer = tiktoken.get_encoding("gpt2")

text1 = "This is a sample text: with numbers 123, special characters !@# and a few more."
text2 = " And a few more Recepie words and Ingredients Apple, sugar, honey car."

text = " <|endoftext|> ".join([text1, text2])


encoded3 = tiktokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(encoded3) 

decoded3 = tiktokenizer.decode(encoded3)
print(decoded3) 



[1212, 318, 257, 6291, 2420, 25, 351, 3146, 17031, 11, 2041, 3435, 5145, 41573, 290, 257, 1178, 517, 13, 220, 50256, 220, 843, 257, 1178, 517, 19520, 21749, 2456, 290, 33474, 4196, 11, 7543, 11, 12498, 1097, 13]
This is a sample text: with numbers 123, special characters !@# and a few more. <|endoftext|>  And a few more Recepie words and Ingredients Apple, sugar, honey car.


# Daten sampling und Kontext-Fenster

- Das Modell wird darauf trainiert immer nur das nächste Token/ Die nächste Token ID zu erraten
- Auf Basis des Kontextfensters wird berrechnet welche Token ID am wahrscheinlichsten als nächstes kommt
- Wir trainieren das Modell darauf, mit einem Input x das wahrscheinlichste Ziel y zu bekommen 

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

- Als erstes unsere Rezepte zu Token IDs encoden

In [18]:
recepieTokens = tiktokenizer.encode(raw_text)

print(recepieTokens[:20])
print(f"Anzahl der Tokens: {len(recepieTokens)}")

[3041, 344, 21749, 25, 1400, 12, 33, 539, 11959, 45305, 198, 41222, 25, 16, 269, 13, 14245, 11856, 7586, 7543]
Anzahl der Tokens: 129872


- Für jeden Textabschitt brauchen wir jeweils den Input und das Ziel
- Da das Modell später das nächste Wort vorhersagen soll muss der Input, sowie das Ziel um ein Wort verschoben sein
- Die größe des Inputs ist das Kontextfenster

In [24]:
context_size = 5

x = recepieTokens[:context_size]
y = recepieTokens[1:context_size+1]

print("x:", x)
print("y:      ", y)

x: [3041, 344, 21749, 25, 1400]
y:       [344, 21749, 25, 1400, 12]


- Als nächstes wird ein Data Loader benötigt, der über den Datensatz itteriert und Inputs (x) sowie Ziele (y) zurückgibt

In [25]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0


- Datensatz und Data Loader erstellen die aus dem Input Text einzelne überlappende Chuncks erstellen

In [27]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [28]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- Mit dem Data Loader Funktion können wir jetzt unseren Rohtext in einzelne überlappende Token ID Chuncks umwandeln (vermischen) und in einzelnen Batches zusammenfassen je nachdem wie hoch der stride ist überlappen sich Input und Target innerhalb der Batches --> zu viel Überlappung zwischen den Batches füht zu overfitting

In [48]:
#dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
#dataloader = create_dataloader_v1(raw_text, batch_size=4, max_length=5, stride=5, shuffle=False)
#dataloader = create_dataloader_v1(raw_text, batch_size=3, max_length=4, stride=1, shuffle=False)
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=1, shuffle=False)

print("Anzahl der Batches:", len(dataloader))

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Anzahl der Batches: 64934
Inputs:
 tensor([[ 3041,   344, 21749,    25],
        [  344, 21749,    25,  1400]])

Targets:
 tensor([[  344, 21749,    25,  1400],
        [21749,    25,  1400,    12]])


# Token Embeddings erstellen

- Mithilfe einer Embedding Layer, werden Tokens als Vektoren dargestellt.
- Die Embeddings werden vor dem Training mit Zufallszahlen erstell und während dem Training ständig so aktuallisiert/verschoben, dass sie Sinn ergeben
- Wenn zwei Vektoren im mathematischen Raum nah beieinander liegen, bedeutet das für das Modell, dass diese Wörter in einem ähnlichen Kontext vorkommen
- Die Embedding Layer ist eine Art Lookup-Table bei dem mit vorhandenen Input IDs direkt die entsprechenden Vektoren ausgegeben werden können

Beispiel: 
- Nach der dem erstellen der Token IDs haben wir die Input Tokens: 2, 7, 5, 3, 9, 8, 3
- Wir haben ein Vokabular an 7 Wörtern und wollen Vektoren mit 5 Dimensionen erstellen
- Das Ergebnis ist eine 7x5 Matrix

In [ ]:
#Ids ins richtige format bringen
input_ids = torch.tensor([2, 7, 5, 3, 8, 9, 3])

#Vocab size und Dimension festlegen
vocab_size = 10
dimension = 5

# Embedding Layer erstellen
embedding_layer = torch.nn.Embedding(vocab_size, dimension)
print(embedding_layer.weight)

# Embeddings für die Input-IDs ausgeben
embeddings = embedding_layer(input_ids)
print(embeddings)


Parameter containing:
tensor([[-0.7001, -0.9466,  0.3888,  0.9234, -1.6390],
        [ 0.6707, -0.3626, -0.2306,  1.4910, -0.7744],
        [-0.8925,  0.8399, -1.3619, -0.3863, -1.0947],
        [ 0.1813, -0.3198, -0.3244, -1.0811, -0.4507],
        [-0.1906, -0.0865, -0.3620, -0.5398,  0.0956],
        [ 0.7806, -0.6742, -0.9443, -1.8795, -0.5478],
        [ 0.8991,  0.9048, -2.2444,  0.4388,  1.1988],
        [ 0.4012,  0.5696,  0.4090, -1.2766, -0.3908],
        [-0.3358,  0.2478, -0.1954,  0.2200, -0.0686],
        [-2.2460,  0.4470, -0.5431,  1.5086, -1.6327]], requires_grad=True)
tensor([[-0.8925,  0.8399, -1.3619, -0.3863, -1.0947],
        [ 0.4012,  0.5696,  0.4090, -1.2766, -0.3908],
        [ 0.7806, -0.6742, -0.9443, -1.8795, -0.5478],
        [ 0.1813, -0.3198, -0.3244, -1.0811, -0.4507],
        [-0.3358,  0.2478, -0.1954,  0.2200, -0.0686],
        [-2.2460,  0.4470, -0.5431,  1.5086, -1.6327],
        [ 0.1813, -0.3198, -0.3244, -1.0811, -0.4507]],
       grad_fn=<Embed

# Positionen der Wörter mit kodieren

- Die Ebedding Layer kodiert die Token-IDs in Vektor
- Die Information an welcher stelle die IDs stehen geht dabei aber verloren
- Die Lösung besteht darin die Vektoren für die Token IDs noch mit Positions-Vektoren zu multiplizieren um die Position der Wörter/Tokens im LLM-Input beizubehalten

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- Der BytePair Encoder hat eine Vokabulargröße von 50,257
- Geben wir unseren IDs Vektoren mit 256 Dimensionen

In [59]:
vocab_size = 50257
dimension = 256

embedding_layer = torch.nn.Embedding(vocab_size, dimension)

-  Mithilfe des Dataloaders können wir Sampel in 8er Batches mit jeweils 4 Tokens erstellen

In [60]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4,
    stride=4, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[ 3041,   344, 21749,    25],
        [ 1400,    12,    33,   539],
        [11959, 45305,   198, 41222],
        [   25,    16,   269,    13],
        [14245, 11856,  7586,  7543],
        [   11,   352,    14,    17],
        [  269,    13, 28959,   515],
        [ 7545,    11,   352,    14]])

Inputs shape:
 torch.Size([8, 4])


- Diese Input Token IDs können wir jetzt in unserer Embedding Layer finden

In [62]:
token_embeddings = embedding_layer(inputs)
print(token_embeddings.shape)

print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 2.2224, -2.2297,  0.5960,  ..., -0.0075,  0.1601, -1.0266],
         [ 1.1921, -0.9988, -0.9123,  ...,  0.4699,  0.5366,  1.0069],
         [-1.0272, -2.4459, -1.1245,  ..., -1.5396, -0.9565, -0.4889],
         [-0.2387, -0.0177,  0.8143,  ...,  1.5704, -0.1626,  1.1252]],

        [[-0.2122, -0.5809, -0.1333,  ...,  0.6184,  0.8834, -1.4241],
         [ 0.9132, -1.1361,  0.1869,  ...,  0.4290,  0.3582, -0.1596],
         [ 0.6712,  0.6696, -0.8492,  ...,  0.9280, -1.1937, -3.7931],
         [-0.5810,  1.5097, -0.6506,  ...,  0.7190, -0.2909, -0.7015]],

        [[ 0.3289,  1.6152, -0.1040,  ..., -1.4709, -0.4554,  0.2464],
         [ 0.0497, -0.7209,  0.8701,  ..., -0.4834, -0.5379, -0.1844],
         [ 2.5245,  0.4188, -0.7118,  ..., -0.3790, -0.8429,  0.1188],
         [-0.3431, -0.1029,  0.5529,  ..., -0.8886,  0.3450,  1.1256]],

        ...,

        [[-0.2293,  0.5384,  0.6784,  ..., -0.1126, -1.4545,  0.1875],
         [-1.8655, -0.4944,  1.38

- Als nächstes wird eine weitere Positional Embedding Layer erstellt

In [67]:
pos_embedding_layer = torch.nn.Embedding(4, dimension)
print(pos_embedding_layer.weight)

pos_embeddings = pos_embedding_layer(torch.arange(4))
print(pos_embeddings.shape)
print(pos_embeddings)


input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(input_embeddings)

Parameter containing:
tensor([[-0.4639, -0.6516, -1.1366,  ...,  0.1126,  0.3610,  0.3823],
        [-0.2923, -2.0659, -0.0040,  ...,  0.5702,  1.0021,  1.4321],
        [ 0.0740, -0.1267, -0.0530,  ...,  0.1731, -0.5546, -0.5839],
        [-0.1625, -0.1949,  0.2764,  ..., -0.7149,  0.2509,  1.0400]],
       requires_grad=True)
torch.Size([4, 256])
tensor([[-0.4639, -0.6516, -1.1366,  ...,  0.1126,  0.3610,  0.3823],
        [-0.2923, -2.0659, -0.0040,  ...,  0.5702,  1.0021,  1.4321],
        [ 0.0740, -0.1267, -0.0530,  ...,  0.1731, -0.5546, -0.5839],
        [-0.1625, -0.1949,  0.2764,  ..., -0.7149,  0.2509,  1.0400]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([8, 4, 256])
tensor([[[ 1.7586e+00, -2.8813e+00, -5.4061e-01,  ...,  1.0508e-01,
           5.2112e-01, -6.4435e-01],
         [ 8.9981e-01, -3.0647e+00, -9.1626e-01,  ...,  1.0402e+00,
           1.5387e+00,  2.4390e+00],
         [-9.5320e-01, -2.5726e+00, -1.1775e+00,  ..., -1.3665e+00,
          -1.5111e+00, -1.072